# 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.

- 채점 기준:
    - 출력 결과 일치: 제출한 코드가 오류 없이 제시된 출력 결과와 일치하거나 맥락상 동일해야 정답으로 인정됩니다.
    - 코드의 다양성 인정: 출력 결과가 맥락상 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# 10주차 과제


In [ ]:
!pip install -q langchain langchain-core langchain-openai langchain-google-genai langgraph tavily-python python-dotenv


In [1]:
# API 키 설정
import os

# 로컬 환경: 프로젝트 루트의 .env 파일에 키를 저장하고 아래 코드를 사용
from dotenv import load_dotenv
load_dotenv()

# Colab 환경: 위 두 줄을 주석 처리하고 아래 코드를 활성화
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")


True

In [2]:
# 공통 설정
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

model = init_chat_model('openai:gpt-4o-mini')


---
## 파트 1. 서브에이전트 패턴


In [ ]:
# 주어진 코드 (수정 불필요)
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool

class ModelCallLogger(AgentMiddleware):
    def __init__(self, agent_name: str):
        self.agent_name = agent_name
        self._call_count = 0
    def before_model(self, state, runtime):
        self._call_count += 1
        print(f'  [LOG] {self.agent_name} | 모델 호출 #{self._call_count}')
    def after_model(self, state, runtime):
        last = state['messages'][-1]
        if getattr(last, 'tool_calls', None):
            print(f'  [LOG] {self.agent_name} | 툴 호출: {[tc["name"] for tc in last.tool_calls]}')
        else:
            content = last.content
            if isinstance(content, list):
                content = content[0].get('text', '') if content else ''
            print(f'  [LOG] {self.agent_name} | 응답: {str(content)[:80]}')

@tool
def get_service_status(service_name: str) -> dict:
    '''사내 서비스의 현재 가동 상태와 장애 여부를 조회합니다.'''
    db = {
        'VPN':  {'status': '장애', 'level': '심각', 'affected_users': 210},
        'Jira': {'status': '점검', 'level': '경고', 'affected_users': 85},
        'Slack': {'status': '정상', 'level': '-', 'affected_users': 0},
    }
    return db.get(service_name, {'status': '알 수 없음'})

@tool
def get_account_info(username: str) -> dict:
    '''사용자의 계정 상태와 보유 권한 목록을 조회합니다.'''
    db = {
        'park.new': {'status': '비활성', 'permissions': [], 'role': '신규 입사자'},
        'kim.dev':  {'status': '활성',   'permissions': ['GitHub', 'Jira'], 'role': '개발자'},
    }
    return db.get(username, {'status': '없음', 'permissions': []})

@tool
def get_access_request_status(username: str) -> dict:
    '''사용자가 신청한 권한 요청의 처리 현황을 조회합니다.'''
    db = {
        'park.new': {'pending': ['Slack', 'GitHub'], 'note': 'HR 승인 대기 중'},
    }
    return db.get(username, {'pending': [], 'note': '진행 중인 요청 없음'})
# ─────────────────────────────────────────────────────────────────────────────


---
## 문제 1. 서브에이전트 패턴 — 전문 서브에이전트 생성

다음 조건을 만족하도록 코드를 완성하시오.
- `system_agent`: `get_service_status` 툴만 사용, 시스템 프롬프트는 IT 인프라 전문가
- `account_agent`: `get_account_info`, `get_access_request_status` 툴 사용, 계정 관리 전문가
- 두 에이전트 모두 `ModelCallLogger` 미들웨어를 적용한다.

**출력 예시**:
```
system_agent type: <class 'langgraph.graph.state.CompiledStateGraph'>
account_agent type: <class 'langgraph.graph.state.CompiledStateGraph'>
```


In [ ]:
# TODO 1: system_agent 생성 (get_service_status 툴, ModelCallLogger 미들웨어)
system_tools = [get_service_status]
system_logger = ModelCallLogger('SystemAgent')

system_agent = create_agent(
    model,
    tools=system_tools,
    system_prompt=(
        '당신은 사내 IT 인프라 전문가입니다. '
        '서비스 장애 현황을 조회해 정확한 상태를 보고하세요.'
    ),
    middleware=[system_logger]
)

# TODO 2: account_agent 생성 (get_account_info, get_access_request_status 툴)
account_tools = [get_account_info, get_access_request_status]
account_logger = ModelCallLogger('AccountAgent')

account_agent = create_agent(
    model,
    tools=account_tools,
    system_prompt=(
        '당신은 사내 계정 및 권한 관리 전문가입니다. '
        '계정 상태, 보유 권한, 권한 요청 현황을 조회해 보고하세요.'
    ),
    middleware=[account_logger]
)

print(f"system_agent type: {type(system_agent)}")
print(f"account_agent type: {type(account_agent)}")

---
## 문제 2. 서브에이전트 패턴 — 고수준 툴 래핑

다음 조건을 만족하도록 코드를 완성하시오.
- `check_system_status(request: str) -> str`: `system_agent`를 호출해 결과를 반환하는 `@tool`
- `check_user_account(request: str) -> str`: `account_agent`를 호출해 결과를 반환하는 `@tool`
- 두 툴의 이름을 출력한다.

**출력 예시**:
```
고수준 툴: ['check_system_status', 'check_user_account']
```


In [ ]:
from langchain.tools import tool
from langchain_core.messages import HumanMessage

# TODO 1: check_system_status — system_agent를 호출하는 @tool
@tool
def check_system_status(request: str) -> str:
    '''사내 서비스의 장애 여부와 운영 상태를 조회합니다.'''
    result = system_agent.invoke({'messages': [HumanMessage(content=request)]})
    return result['messages'][-1].text

# TODO 2: check_user_account — account_agent를 호출하는 @tool
@tool
def check_user_account(request: str) -> str:
    '''사용자 계정 상태와 권한 현황을 조회합니다.'''
    result = account_agent.invoke({'messages': [HumanMessage(content=request)]})
    return result['messages'][-1].text

print('고수준 툴:', [t.name for t in [check_system_status, check_user_account]])

---
## 문제 3. 서브에이전트 패턴 — 슈퍼바이저 생성

다음 조건을 만족하도록 코드를 완성하시오.
- `helpdesk_agent`: `check_system_status`, `check_user_account` **고수준 툴만** 사용하는 슈퍼바이저
- `ModelCallLogger('HelpdeskAgent')` 미들웨어 적용

**출력 예시**:
```
슈퍼바이저: <langgraph.graph.state.CompiledStateGraph object at 0x113554510>
```


In [ ]:
# TODO: helpdesk_agent 생성 (고수준 툴만, HelpdeskAgent 로거)

supervisor_tools = [check_system_status, check_user_account]
helpdesk_agent = create_agent(
    model,
    tools=supervisor_tools,
    system_prompt=(
        '당신은 사내 IT 헬프데스크 담당자입니다. '
        '서비스 장애인지 계정 문제인지 판단해 적절한 툴을 호출하세요.'
    ),
    middleware=[ModelCallLogger('HelpdeskAgent')]
)

print('슈퍼바이저:', helpdesk_agent)

---
## 파트 2. 핸드오프 패턴


In [10]:
# 주어진 코드 (수정 불필요)

from typing import Literal
from typing_extensions import NotRequired, Annotated
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.types import Command

@tool
def approve_claim(reason: str) -> str:
    '''보험 청구를 승인합니다.'''
    return f'청구 승인 완료. 사유: {reason}'

@tool
def reject_claim(reason: str) -> str:
    '''보험 청구를 거절합니다.'''
    return f'청구 거절. 사유: {reason}'


---
## 문제 4. 핸드오프 패턴 — ClaimState와 Command 반환 툴

다음 조건을 만족하도록 코드를 완성하시오.
- `ClaimState`: `AgentState`를 상속하며 `current_step`, `claim_type`, `damage_amount` 필드를 가진다.
  - 세 필드 모두 `NotRequired`를 적용한다.
- `record_claim_type`: `claim_type`을 기록하고 `damage_assessment` 단계로 전환하는 `@tool`
  - `Command`를 반환하여 상태를 업데이트한다.
- assert 에서 오류가 발생하지 않는다.



In [ ]:
import inspect

# TODO 1: ClaimState 정의 (AgentState 상속, NotRequired 적용)
ClaimStep = Literal['claim_intake', 'damage_assessment', 'compensation_decision']

class ClaimState(AgentState):
    current_step: NotRequired[ClaimStep]
    claim_type:   NotRequired[str]
    damage_amount: NotRequired[str]

# TODO 2: record_claim_type 툴 정의
@tool
def record_claim_type(
    claim_type: Literal['자동차', '화재', '상해', '도난', '기타'],
    runtime: ToolRuntime[None, ClaimState],
) -> Command:
    '청구 유형을 기록하고 손해 평가 단계로 전환합니다.'
    msg = ToolMessage(
        content=f'청구 유형 기록 완료: {claim_type}',
        tool_call_id=runtime.tool_call_id,
    )
    return Command(update={
        'messages': [msg],
        'claim_type': claim_type,
        'current_step': 'damage_assessment',
    })

@tool
def record_damage_amount(
    amount: str,
    runtime: ToolRuntime[None, 'ClaimState'],
) -> Command:
    '''피해 금액을 기록하고 보상 결정 단계로 전환합니다.'''
    return Command(update={
        'messages': [ToolMessage(content=f'피해 금액 평가 완료: {amount}',
                                 tool_call_id=runtime.tool_call_id)],
        'damage_amount': amount,
        'current_step': 'compensation_decision',
    })


# 검증
assert 'current_step' in ClaimState.__annotations__
assert 'claim_type' in ClaimState.__annotations__
assert 'damage_amount' in ClaimState.__annotations__

---
## 문제 5. 핸드오프 패턴 — STEP_CONFIG와 apply_step_config 미들웨어

다음 조건을 만족하도록 코드를 완성하시오.
- `STEP_CONFIG`: 세 단계(`claim_intake`, `damage_assessment`, `compensation_decision`)의 툴, 필수 상태 키 정의
- `apply_step_config`: `@wrap_model_call` 데코레이터를 사용해 `current_step`에 따라 시스템 프롬프트와 툴을 동적으로 교체

**출력 예시**:
```
STEP_CONFIG 단계: ['claim_intake', 'damage_assessment', 'compensation_decision']
  claim_intake: 툴=['record_claim_type'], requires=[]
  damage_assessment: 툴=['record_damage_amount'], requires=['claim_type']
  compensation_decision: 툴=['approve_claim', 'reject_claim'], requires=['claim_type', 'damage_amount']
```


In [ ]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# TODO 1: STEP_CONFIG 딕셔너리 정의
STEP_CONFIG = {
    'claim_intake': {
        'prompt': (
            '당신은 보험 접수 담당자입니다.\n'
            '고객의 사고 유형을 파악하고 record_claim_type을 호출하세요.'
        ),
        'tools': [record_claim_type],
        'requires': [],
    },
    'damage_assessment': {
        'prompt': (
            '당신은 손해 사정 전문가입니다.\n'
            '청구 유형: {claim_type}\n'
            '피해 금액을 확인하고 record_damage_amount를 호출하세요.'
        ),
        'tools': [record_damage_amount],
        'requires': ['claim_type'],
    },
    'compensation_decision': {
        'prompt': (
            '당신은 보상 심사 담당자입니다.\n'
            '청구 유형: {claim_type} | 피해 금액: {damage_amount}\n'
            '자동차·화재·상해이고 금액이 합리적이면 approve_claim,\n'
            '도난이거나 금액 불명확하면 reject_claim을 호출하세요.'
        ),
        'tools': [approve_claim, reject_claim],
        'requires': ['claim_type', 'damage_amount'],
    },
}

def _format_step_prompt(prompt_template: str, state: dict, required_keys: list) -> str:
    '''단계별 프롬프트를 상태 값으로 포매팅합니다.'''
    format_kwargs = {k: state.get(k, '') for k in required_keys}
    return prompt_template.format(**format_kwargs)

# TODO 2: apply_step_config 미들웨어 구현
@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    current_step = request.state.get('current_step', 'claim_intake')
    cfg = STEP_CONFIG[current_step]
    for key in cfg['requires']:
        if request.state.get(key) is None:
            raise ValueError(f"'{key}' 가 설정되지 않았습니다.")
    formatted_prompt = _format_step_prompt(cfg['prompt'], request.state, cfg['requires'])
    return handler(request.override(
        system_prompt=formatted_prompt,
        tools=cfg['tools'],
    ))

# 검증
print('STEP_CONFIG 단계:', list(STEP_CONFIG.keys()))
for step, cfg in STEP_CONFIG.items():
    print(f'  {step}: 툴={[t.name for t in cfg["tools"]]}, requires={cfg["requires"]}')

---
## 파트 3. 스킬즈 패턴


In [ ]:
# ─── 주어진 코드 (수정 불필요) ───────────────────────────────────────────
SKILLS_DATA = [
    {
        'name': 'vacation_policy',
        'description': '연차·병가·특별휴가 발생 기준, 사용 절차, 잔여 일수 조회 방법.',
        'content': '# 휴가 정책\n\n## 연차휴가\n- 입사 1년 미만: 매월 1일 발생 (최대 11일)\n- 1년 이상: 15일 기본',
    },
    {
        'name': 'expense_policy',
        'description': '출장·식대·교육비 경비 처리 한도, 영수증 제출 절차, 승인 단계.',
        'content': '# 경비 처리 정책\n\n## 식대\n- 업무 식사: 1인당 20,000원 (3인 이상)',
    },
    {
        'name': 'remote_work_policy',
        'description': '재택근무 신청 조건·절차, 장비 지원 기준, 보안 준수 사항.',
        'content': '# 재택근무 정책\n\n## 신청 조건\n- 입사 후 3개월 이상 경과한 정규직만 신청 가능',
    },
]
# ─────────────────────────────────────────────────────────────────────────────


---
## 문제 6. 스킬즈 패턴 — Skill 구조와 load_skill 툴

다음 조건을 만족하도록 코드를 완성하시오.
- `SKILLS`: `SKILLS_DATA`로 초기화한 `list[Skill]` 변수
- `load_skill(skill_name: str) -> str`: 이름으로 스킬을 검색해 전체 내용을 반환하는 `@tool`
  - 없으면 `'사용 가능한 스킬: ...'` 안내 문자열 반환

**출력 예시**:
```
vacation_policy 로드: 스킬 로드 완료: vacation_policy
...

없는 스킬: 'unknown' 스킬을 찾을 수 없습니다. 사용 가능한 스킬: vacation_policy, expense_policy, remote_work_policy
```


In [ ]:
from typing import TypedDict
from langchain.tools import tool

class Skill(TypedDict):
    name: str
    description: str
    content: str

# TODO 1: SKILLS_DATA를 list[Skill]로 초기화
SKILLS: list[Skill] = list(SKILLS_DATA)  # SKILLS_DATA items already match Skill structure

# TODO 2: load_skill @tool 구현
@tool
def load_skill(skill_name: str) -> str:
    '''스킬의 전체 내용을 에이전트 컨텍스트에 로드합니다.'''
    matching = [s for s in SKILLS if s['name'] == skill_name]
    if matching:
        skill = matching[0]
        return f'스킬 로드 완료: {skill_name}\n\n{skill["content"]}'
    available = ', '.join(s['name'] for s in SKILLS)
    return f"'{skill_name}' 스킬을 찾을 수 없습니다. 사용 가능한 스킬: {available}"

# 검증
result = load_skill.invoke({'skill_name': 'vacation_policy'})
print('vacation_policy 로드:', result[:100])
print("-" * 60)
result_miss = load_skill.invoke({'skill_name': 'unknown'})
print('없는 스킬:', result_miss)

---
## 문제 7. 스킬즈 패턴 — SkillMiddleware 구현

다음 조건을 만족하도록 `SkillMiddleware`를 완성하시오.
- `AgentMiddleware`를 상속한다.
- 클래스 변수 `tools = [load_skill]`을 선언해 `create_agent`가 툴을 자동 등록하게 한다.
- `__init__`: SKILLS의 `name`과 `description`으로 스킬 안내 프롬프트 문자열(`skills_prompt`)을 생성한다.
- `wrap_model_call`: 시스템 메시지 `content_blocks`에 스킬 목록 텍스트를 추가한다.

**출력 예시**:
```
등록된 툴: ['load_skill']
vacation_policy 포함 여부: True
```


In [ ]:
from typing import Callable
from langchain.agents.middleware import AgentMiddleware, ModelRequest, ModelResponse
from langchain.messages import SystemMessage

# TODO: SkillMiddleware 클래스 완성
class SkillMiddleware(AgentMiddleware):           # TODO 1: 상속할 클래스
    tools = [load_skill]                          # TODO 2: create_agent에 툴을 자동 등록하는 클래스 변수

    def __init__(self):
        lines = []
        for skill in SKILLS:
            lines.append(f'- **{skill["name"]}**: {skill["description"]}')
        self.skills_prompt = '\n'.join(lines)     # TODO 3: 스킬 목록 문자열 저장 변수명

    def wrap_model_call(                           # TODO 4: 메서드 이름
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        skills_addendum = (
            f'\n\n## 사용 가능한 스킬\n\n{self.skills_prompt}\n\n'
            '특정 정책의 상세 내용이 필요하면 load_skill 툴을 호출하세요.'
        )
        existing_blocks = list(request.system_message.content_blocks)  # TODO 5
        existing_blocks.append({'type': 'text', 'text': skills_addendum})
        new_system_message = SystemMessage(content=existing_blocks)
        modified_request = request.override(system_message=new_system_message)
        return handler(modified_request)

# 검증
print('등록된 툴:', [t.name for t in SkillMiddleware.tools])
mw = SkillMiddleware()
print('vacation_policy 포함 여부:', 'vacation_policy' in mw.skills_prompt)

---
## 파트 4. 라우터 패턴


In [16]:
# 주어진 코드 (수정 불필요)

import operator
from typing import Annotated, Literal, TypedDict
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

router_llm = init_chat_model('openai:gpt-4o-mini')

@tool
def get_product_info(product_name: str) -> dict:
    '''상품의 스펙, 가격, 재고 여부를 조회합니다.'''
    catalog = {
        '노트북 A': {'price': 1_200_000, 'spec': 'Intel i5, RAM 16GB', 'stock': '재고 있음'},
        '스마트워치 C': {'price': 320_000, 'spec': 'GPS, 방수 5ATM', 'stock': '재고 있음'},
    }
    return catalog.get(product_name, {'error': f"'{product_name}' 상품 정보 없음"})

@tool
def get_order_status(order_id: str) -> dict:
    '''주문 번호로 현재 주문 상태와 배송 현황을 조회합니다.'''
    orders = {
        'ORD-1001': {'product': '노트북 A', 'status': '배송 중', 'carrier': 'CJ대한통운'},
    }
    return orders.get(order_id, {'error': f"'{order_id}' 주문 없음"})

@tool
def get_product_reviews(product_name: str) -> dict:
    '''상품의 최근 구매 후기와 평균 평점을 조회합니다.'''
    reviews = {
        '노트북 A': {'avg_rating': 4.5, 'summary': '성능 우수, 발열 적음'},
        '스마트워치 C': {'avg_rating': 4.7, 'summary': '건강 측정 정확, 디자인 호평'},
    }
    return reviews.get(product_name, {'error': f"'{product_name}' 리뷰 없음"})

product_agent = create_agent(
    router_llm, tools=[get_product_info],
    system_prompt='당신은 쇼핑몰 상품 전문가입니다. 상품 스펙, 가격, 재고를 조회해 답하세요.',
)
order_agent = create_agent(
    router_llm, tools=[get_order_status],
    system_prompt='당신은 쇼핑몰 주문·배송 전문가입니다. 주문 상태와 배송 현황을 조회해 답하세요.',
)
review_agent = create_agent(
    router_llm, tools=[get_product_reviews],
    system_prompt='당신은 쇼핑몰 리뷰 전문가입니다. 구매 후기와 평균 평점을 조회해 답하세요.',
)


---
## 문제 8. 라우터 패턴 — RouterState와 분류 스키마 정의

다음 조건을 만족하도록 코드를 완성하시오.
- `AgentOutput`: `source(str)`, `result(str)` 필드를 가진 `TypedDict`
- `Classification`: `source(Literal['product', 'order', 'review'])`, `query(str)` 필드를 가진 `TypedDict`
- `RouterState`: `query`, `classifications`, `results`, `final_answer` 필드를 가진 `TypedDict`
  - `results`는 여러 에이전트가 병렬로 반환하는 결과를 자동 병합하도록 `Annotated`와 `operator.add`를 적용한다.
- `ClassificationResult`: Pydantic `BaseModel`, `classifications: list[Classification]` 필드
- `classify_query(state)`: `router_llm.with_structured_output(ClassificationResult)`로 쿼리를 분류한다.

**출력 예시**:
```
RouterState 필드: ['query', 'classifications', 'results', 'final_answer']
분류 결과: ['product']
```


In [ ]:
from pydantic import BaseModel, Field
from langgraph.graph import END, START, StateGraph

# TODO 1: AgentOutput TypedDict 정의
class AgentOutput(TypedDict):
    source: str
    result: str

# TODO 2: Classification TypedDict 정의
class Classification(TypedDict):
    source: Literal['product', 'order', 'review']
    query: str

# TODO 3: RouterState TypedDict 정의
class RouterState(TypedDict):
    query: str
    classifications: list[Classification]
    results: Annotated[list[AgentOutput], operator.add]
    final_answer: str

# TODO 4: ClassificationResult Pydantic 모델 정의
class ClassificationResult(BaseModel):
    classifications: list[Classification] = Field(
        description='분류된 에이전트 목록과 각 에이전트에 전달할 서브 질문'
    )

# TODO 5: classify_query 노드 구현
def classify_query(state: RouterState) -> dict:
    '쿼리를 분석하고 어떤 에이전트를 호출할지 결정합니다.'
    classifier = router_llm.with_structured_output(ClassificationResult)
    classification_result = classifier.invoke([
        {
            'role': 'system',
            'content': (
                '이 쿼리를 분석해 관련 에이전트를 선택하세요.\n'
                '- product : 상품 스펙, 가격, 재고\n'
                '- order   : 주문 상태, 배송 현황\n'
                '- review  : 구매 후기, 평균 평점'
            ),
        },
        {'role': 'user', 'content': state['query']},
    ])
    return {'classifications': classification_result.classifications}

# 검증
print('RouterState 필드:', list(RouterState.__annotations__.keys()))
test = classify_query({'query': '노트북 A 가격 알려주세요.'})
print('분류 결과:', [c['source'] for c in test['classifications']])

---
## 문제 9. 라우터 패턴 — Send 팬아웃과 에이전트 노드 구현

다음 조건을 만족하도록 코드를 완성하시오.
- `route_to_agents(state)`: `classifications`를 순회하며 `Send` 리스트를 반환한다.
  - `Send(에이전트_노드_이름, {'query': 서브_질문})`
- `query_product(state)`, `query_order(state)`, `query_review(state)`: 각 전문 에이전트를 호출하고 `{'results': [...]}` 를 반환한다.
  - 반환 딕셔너리의 `results`는 `[{'source': '에이전트명', 'result': '응답'}]` 형태

**출력 예시**:
```
Send 팬아웃 테스트: [Send('product', ...), Send('review', ...)]
```


In [ ]:
from langgraph.types import Send

# TODO 1: route_to_agents — classifications를 Send 리스트로 변환
def route_to_agents(state: RouterState) -> list[Send]:
    '분류 결과를 Send로 변환해 병렬 팬아웃합니다.'
    sends = []
    for c in state['classifications']:
        sends.append(Send(c['source'], {'query': c['query']}))
    return sends

# 에이전트 매핑 딕셔너리
_AGENT_MAP = {
    'product': product_agent,
    'order': order_agent,
    'review': review_agent,
}

def _run_agent(agent_name: str, state) -> dict:
    '''공통 에이전트 실행 헬퍼.'''
    agent = _AGENT_MAP[agent_name]
    result = agent.invoke({'messages': [{'role': 'user', 'content': state['query']}]})
    return {'results': [{'source': agent_name, 'result': result['messages'][-1].content}]}

# TODO 2: query_product — product_agent 호출, results 반환
def query_product(state) -> dict:
    return _run_agent('product', state)

# TODO 3: query_order — order_agent 호출
def query_order(state) -> dict:
    return _run_agent('order', state)

# TODO 4: query_review — review_agent 호출
def query_review(state) -> dict:
    return _run_agent('review', state)

# 검증 (API 불필요)
test_state = {'classifications': [
    {'source': 'product', 'query': '노트북 A 가격?'},
    {'source': 'review',  'query': '노트북 A 후기?'},
]}
sends = route_to_agents(test_state)
print('Send 팬아웃 테스트:', sends)

---
## 문제 10. 라우터 패턴 — 결과 종합 노드와 전체 워크플로우 조립

다음 조건을 만족하도록 코드를 완성하시오.
- `synthesize_results(state)`: 수집된 모든 결과를 `router_llm`으로 종합해 `final_answer`를 반환한다.
  - `state['results']`가 비어 있으면 `'어떤 에이전트에서도 결과를 찾지 못했습니다.'`를 반환한다.
- `StateGraph(RouterState)`로 워크플로우를 조립한다.
  - 노드: `classify`, `product`, `order`, `review`, `synthesize`
  - `classify → (조건부) → product/order/review → synthesize → END`
  - 조건부 팬아웃에 `add_conditional_edges` 사용
- 실행 후 `final_answer`를 출력한다.

**출력 예시**:
```
분류 결과: ['product', 'review']

[노트북 A에 대한 상품 정보와 후기를 종합한 답변]
```


In [ ]:
# TODO 1: synthesize_results 노드 구현
def synthesize_results(state: RouterState) -> dict:
    '수집된 모든 결과를 하나의 일관된 응답으로 종합합니다.'
    if not state['results']:
        return {'final_answer': '어떤 에이전트에서도 결과를 찾지 못했습니다.'}

    formatted = [
        f'**{r["source"].upper()} 조회 결과:**\n{r["result"]}'
        for r in state['results']
    ]
    response = router_llm.invoke([
        {
            'role': 'system',
            'content': (
                f'다음 조회 결과를 종합해 원래 질문에 답하세요: "{state["query"]}"\n\n'
                '여러 소스 정보를 중복 없이 통합하고 친절하게 답변하세요.'
            ),
        },
        {'role': 'user', 'content': '\n\n'.join(formatted)},
    ])
    return {'final_answer': response.content}

# TODO 2: 전체 워크플로우 StateGraph 조립
graph_builder = StateGraph(RouterState)

# 노드 등록
for name, func in [('classify', classify_query), ('product', query_product),
                    ('order', query_order), ('review', query_review),
                    ('synthesize', synthesize_results)]:
    graph_builder.add_node(name, func)

# 엣지 연결
graph_builder.add_edge(START, 'classify')
graph_builder.add_conditional_edges('classify', route_to_agents, ['product', 'order', 'review'])
for agent_node in ['product', 'order', 'review']:
    graph_builder.add_edge(agent_node, 'synthesize')
graph_builder.add_edge('synthesize', END)

workflow = graph_builder.compile()

# TODO 3: 실행
result = workflow.invoke({'query': '노트북 A 스펙이랑 후기 알려주세요.'})
print('분류 결과:', [c['source'] for c in result['classifications']])
print("-" * 60)
print(result['final_answer'])